# XNLI (Spanish) — NLI with BETO (SOLVED)

**Task:** Natural Language Inference (NLI) in Spanish using the XNLI dataset (subset `es`).  
**Model:** `dccuchile/bert-base-spanish-wwm-cased` (BETO).  
**Goal:** Fine-tune BETO to classify *(premise, hypothesis)* into **entailment**, **neutral**, or **contradiction**.

This notebook is organized as a SOLVED template: fully commented, modular, and ready for Google Colab.
It includes robust installation (with a **Reset Cell 0**), a clear configuration block, training with `Trainer`,
metrics (accuracy, precision, recall, F1 macro), a confusion matrix, error analysis, saving/loading the best model,
and an inference helper.


## Quick start
1. **If running on Colab:** execute **Cell 0** (Reset & Install), then **Runtime → Restart**.
2. Run all cells from **Section 1** onward (skip Cell 0 after restart).
3. Results are saved under `./outputs/` and `./best_*`.


## Cell 0 — Colab Reset & Install (run once, then restart)
This cell avoids NumPy ABI conflicts and pins a consistent, known-good stack.
**Run it once → Restart runtime → then run from Section 1.**


In [ ]:

# ✅ Cell 0 — Reset & Install (Colab)
import sys
pip = f"{sys.executable} -m pip"

!{pip} uninstall -y transformers tokenizers datasets accelerate evaluate simpletransformers numpy
!{pip} install -U "numpy==2.0.2"
!{pip} install -U "pyarrow-hotfix" "fsspec==2024.3.1"
!{pip} install -U "transformers==4.56.2" "datasets==2.19.1" "accelerate==1.10.1" "evaluate==0.4.1" "scikit-learn"

import numpy, transformers, datasets, evaluate, inspect
from transformers import TrainingArguments
print("numpy        :", numpy.__version__)
print("transformers :", transformers.__version__)
print("datasets     :", datasets.__version__)
print("evaluate     :", evaluate.__version__)
print("Has 'evaluation_strategy'?", "evaluation_strategy" in inspect.signature(TrainingArguments.__init__).parameters)

print("\n✅ Done. Now go to: Runtime → Restart. Then continue from Section 1.")


## 1) Setup & Config
- Imports and device
- Reproducibility seed
- Centralized configuration (paths, hyperparameters, run name)


In [ ]:

import os, sys, math, random, json, time
from typing import Tuple

import numpy as np
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, TrainingArguments, Trainer
)

from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report
)
import matplotlib.pyplot as plt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MODEL_NAME = "dccuchile/bert-base-spanish-wwm-cased"
RUN_NAME   = "xnli-es-beto-solved"

LR            = 2e-5
EPOCHS        = 3
BATCH_TRAIN   = 16
BATCH_EVAL    = 32
WEIGHT_DECAY  = 0.01

OUTPUT_DIR    = f"./outputs/{RUN_NAME}"
BEST_DIR      = f"./best_{RUN_NAME}"
os.makedirs(OUTPUT_DIR, exist_ok=True)


## 2) Data — Load XNLI (Spanish subset)
We use the Hugging Face `xnli` dataset with `config='es'` which provides:
- `train`, `validation`, `test`
Each sample has `premise`, `hypothesis`, and `label` (3 classes).


In [ ]:

raw = load_dataset("xnli", "es")
print(raw)
print("train:", len(raw["train"]), "| validation:", len(raw["validation"]), "| test:", len(raw["test"]))
print("Sample:", raw["train"][0])


## 3) Labels & Tokenization
- Ensure the label column is named `labels` (what `Trainer` expects).
- Tokenize `(premise, hypothesis)` pairs with the BETO tokenizer.
- Use dynamic padding via `DataCollatorWithPadding`.


In [ ]:

def add_labels(batch):
    return {"labels": batch["label"]}

raw = raw.map(add_labels)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tokenize_fn(batch):
    out = tokenizer(batch["premise"], batch["hypothesis"], truncation=True)
    out["labels"] = batch["labels"]
    return out

train_ds = raw["train"].map(tokenize_fn, batched=True)
val_ds   = raw["validation"].map(tokenize_fn, batched=True)
test_ds  = raw["test"].map(tokenize_fn, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print("Tokenized features:", train_ds.features)


## 4) Model — BETO for 3-way classification
We load `AutoModelForSequenceClassification` with `num_labels=3` for:
- entailment (0), neutral (1), contradiction (2)


In [ ]:

NUM_LABELS = 3
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
model.to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params/1e6:.2f}M")


## 5) Metrics
We compute **accuracy**, **precision**, **recall**, and **F1 (macro)** using scikit-learn.


In [ ]:

def compute_metrics(eval_pred):
    # Supports tuple or EvalPrediction
    try:
        logits, labels = eval_pred
    except Exception:
        logits, labels = eval_pred.predictions, eval_pred.label_ids

    logits = np.asarray(logits)
    labels = np.asarray(labels)
    preds  = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}


## 6) Training
Evaluate & save each epoch; load best model at the end (based on `f1`). Uses fp16 on GPU if available.


In [ ]:

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_TRAIN,
    per_device_eval_batch_size=BATCH_EVAL,
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    logging_steps=50,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,  # with Transformers 5.0+, prefer 'processing_class=tokenizer'
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
print(train_result)
print("Best checkpoint:", getattr(trainer.state, "best_model_checkpoint", None))

print("Validation metrics:", trainer.evaluate(eval_dataset=val_ds))


## 7) Test Evaluation
Compute test metrics; plot confusion matrix; show per-class report.


In [ ]:

pred = trainer.predict(test_ds)
logits = pred.predictions
y_true = pred.label_ids
y_pred = np.argmax(logits, axis=-1)

acc = accuracy_score(y_true, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
print(f"TEST -> accuracy: {acc:.4f} | precision(macro): {prec:.4f} | recall(macro): {rec:.4f} | f1(macro): {f1:.4f}")

labels_names = ["entailment (0)", "neutral (1)", "contradiction (2)"]
cm = confusion_matrix(y_true, y_pred)

fig = plt.figure(figsize=(5,4))
plt.imshow(cm, interpolation="nearest")
plt.title("Confusion Matrix — XNLI (es)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(np.arange(3), labels_names, rotation=20)
plt.yticks(np.arange(3), labels_names)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, format(cm[i, j], 'd'), ha="center", va="center")
plt.tight_layout()
plt.show()

print(classification_report(y_true, y_pred, target_names=["entailment","neutral","contradiction"], digits=4))


## 8) Error Analysis
Inspect a few misclassified examples to understand typical errors.


In [ ]:

import random
random.seed(SEED)

indices = [i for i, (a,b) in enumerate(zip(y_true, y_pred)) if a != b]
print("Misclassifications:", len(indices))
show_k = min(10, len(indices))
sample_ids = random.sample(indices, show_k) if show_k > 0 else []

for i in sample_ids:
    prem = test_ds[i]["premise"]
    hyp  = test_ds[i]["hypothesis"]
    print("="*80)
    print(f"true={y_true[i]}  pred={y_pred[i]}")
    print("Premise   :", prem)
    print("Hypothesis:", hyp)


## 9) Save & Reload
Save the best model and tokenizer; demonstrate quick inference.


In [ ]:

os.makedirs(BEST_DIR, exist_ok=True)
trainer.save_model(BEST_DIR)
tokenizer.save_pretrained(BEST_DIR)
print("Saved to:", BEST_DIR)

from transformers import AutoTokenizer, AutoModelForSequenceClassification

reload_tok   = AutoTokenizer.from_pretrained(BEST_DIR)
reload_model = AutoModelForSequenceClassification.from_pretrained(BEST_DIR).to(DEVICE)
reload_model.eval()

def infer_nli(premise: str, hypothesis: str) -> Tuple[int, np.ndarray]:
    # Return (pred_id, probs[3]) where labels=[entailment, neutral, contradiction].
    with torch.no_grad():
        enc = reload_tok(premise, hypothesis, truncation=True, return_tensors="pt").to(DEVICE)
        out = reload_model(**enc)
        logits = out.logits.detach().cpu().numpy()[0]
        probs = np.exp(logits - logits.max()) ; probs = probs / probs.sum()
        pred  = int(np.argmax(probs))
        return pred, probs

demo_p = "María compra un libro en la librería."
demo_h = "María adquiere un libro en una tienda."
pred_id, probs = infer_nli(demo_p, demo_h)
print("Demo prediction:", pred_id, "| probs:", probs, "| labels: [entailment, neutral, contradiction]")


## 10) Troubleshooting
- **NumPy ABI / `_center` import errors**: Run **Cell 0**, restart runtime, then run from Section 1.
- **`evaluation_strategy` not recognized**: Your `transformers` is too old. Re-run **Cell 0** to install 4.56.2.
- **`model returned only logits`**: Ensure tokenization adds a `labels` field or set `label_names=['label']` in `Trainer`.
- **Metrics download errors**: This notebook uses scikit-learn metrics to avoid Hub downloads.
- **CUDA OOM**: Lower batch sizes (`per_device_*_batch_size`) or use gradient accumulation.
